In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".csv"):
            print(os.path.join(root, file))

ENSEMBLE (BERT + DistilBERT) - Dataset 1 - BBC Text

1. IMPORTS

In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

print("Imported required libraries...")

2. CONFIGURATION

In [ ]:

BERT_MODEL_NAME = "bert-base-uncased"
DISTILBERT_MODEL_NAME = "distilbert-base-uncased"

LEARNING_RATE = 0.00003
BATCH_SIZE = 16
WEIGHT_DECAY = 0.01

EPOCHS = 5
WARMUP_RATIO = 0.1
MAX_LEN = 128
DROPOUT = 0.1
SEED = 42

print("Defined Parameters...")

3. DATASET PATH

In [ ]:
FILE_NAME = "bbc-text-preprocessed.csv"

file_matches = glob.glob(
    f"/kaggle/input/**/{FILE_NAME}",
    recursive=True
)

if len(file_matches) == 0:
    raise FileNotFoundError(
        f"Could not find {FILE_NAME} inside /kaggle/input/"
    )

NEWS_FILE = file_matches[0]

print("Dataset path:", NEWS_FILE)

4. DEVICE

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

5. SEED

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

6. DATASET CLASS

In [ ]:
class TextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len=MAX_LEN
    ):

        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_len
        )

        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(
                value[idx],
                dtype=torch.long
            )
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item

print("Dataset class defined...")

7. METRICS

In [ ]:
def compute_metrics(pred):

    labels = pred.label_ids
    predictions = pred.predictions

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    preds = np.argmax(
        predictions,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="weighted",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        labels,
        preds
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Metrics function defined...")

8. LOAD DATASET

In [ ]:
print("\n" + "=" * 70)
print("Running ENSEMBLE (BERT + DistilBERT) on: Dataset 1 - BBC Text")
print("=" * 70)

df = pd.read_csv(NEWS_FILE)

print("Original shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

9. DATA CLEANING

In [ ]:
required_columns = [
    "clean_text",
    "category"
]

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df = df.dropna(
    subset=[
        "clean_text",
        "category"
    ]
).copy()

df["clean_text"] = (
    df["clean_text"]
    .astype(str)
    .str.strip()
)

df["category"] = (
    df["category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df = df[
    (df["clean_text"] != "") &
    (df["category"] != "")
].reset_index(drop=True)

print(
    "After removing missing/empty values:",
    df.shape
)

before_duplicates = len(df)

df = df.drop_duplicates(
    subset=["clean_text"]
).reset_index(drop=True)

removed_duplicates = (
    before_duplicates - len(df)
)

print(
    "Duplicate rows removed:",
    removed_duplicates
)

print(
    "Final dataset shape:",
    df.shape
)

10. LABEL ENCODING

In [ ]:
labels_unique = sorted(
    df["category"].unique()
)

label2id = {
    label: idx
    for idx, label in enumerate(labels_unique)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

df["label_id"] = df["category"].map(
    label2id
)

num_labels = len(labels_unique)

print(
    "\nNumber of classes:",
    num_labels
)

print(
    "Classes:",
    labels_unique
)

print(
    "\nClass distribution:"
)

print(
    df["category"].value_counts()
)

print(
    "\nLabel mapping:",
    label2id
)

11. TRAIN / VALIDATION / TEST SPLIT

In [ ]:
train_texts, temp_texts, train_labels, temp_labels = (
    train_test_split(
        df["clean_text"].tolist(),
        df["label_id"].tolist(),
        test_size=0.20,
        random_state=SEED,
        stratify=df["label_id"]
    )
)

val_texts, test_texts, val_labels, test_labels = (
    train_test_split(
        temp_texts,
        temp_labels,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_labels
    )
)

print(
    "\nTraining samples:",
    len(train_texts)
)

print(
    "Validation samples:",
    len(val_texts)
)

print(
    "Testing samples:",
    len(test_texts)
)

12. TOKENIZERS

In [ ]:
bert_tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL_NAME)
distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained(DISTILBERT_MODEL_NAME)


13. CREATE DATASETS

In [ ]:
bert_train_dataset = TextDataset(train_texts, train_labels, bert_tokenizer, MAX_LEN)
bert_val_dataset   = TextDataset(val_texts, val_labels, bert_tokenizer, MAX_LEN)
bert_test_dataset  = TextDataset(test_texts, test_labels, bert_tokenizer, MAX_LEN)

distilbert_train_dataset = TextDataset(train_texts, train_labels, distilbert_tokenizer, MAX_LEN)
distilbert_val_dataset   = TextDataset(val_texts, val_labels, distilbert_tokenizer, MAX_LEN)
distilbert_test_dataset  = TextDataset(test_texts, test_labels, distilbert_tokenizer, MAX_LEN)

print("Datasets created for both models...")

14. TRAIN BERT

In [ ]:
bert_model = BertForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT
)

bert_model.to(device)

bert_training_args = TrainingArguments(

    output_dir="/kaggle/working/Ensemble_BERT",

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    warmup_ratio=WARMUP_RATIO,

    eval_strategy="epoch",

    save_strategy="epoch",
    save_total_limit=1,

    load_best_model_at_end=False,

    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=20,

    optim="adamw_torch",

    seed=SEED,
    data_seed=SEED,

    report_to="none",

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=2
)

bert_trainer = Trainer(

    model=bert_model,

    args=bert_training_args,

    train_dataset=bert_train_dataset,

    eval_dataset=bert_val_dataset,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("\nTraining BERT (LR={}, BS={}, WD={})...".format(
    LEARNING_RATE, BATCH_SIZE, WEIGHT_DECAY))

bert_train_result = bert_trainer.train()

15. TRAIN DISTILBERT

In [ ]:
distilbert_model = DistilBertForSequenceClassification.from_pretrained(
    DISTILBERT_MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dropout=DROPOUT,
    attention_dropout=DROPOUT
)

distilbert_model.to(device)

distilbert_training_args = TrainingArguments(

    output_dir="/kaggle/working/Ensemble_DistilBERT",

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    warmup_ratio=WARMUP_RATIO,

    eval_strategy="epoch",

    save_strategy="epoch",
    save_total_limit=1,

    load_best_model_at_end=False,

    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=20,

    optim="adamw_torch",

    seed=SEED,
    data_seed=SEED,

    report_to="none",

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=2
)

distilbert_trainer = Trainer(

    model=distilbert_model,

    args=distilbert_training_args,

    train_dataset=distilbert_train_dataset,

    eval_dataset=distilbert_val_dataset,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("\nTraining DistilBERT (LR={}, BS={}, WD={})...".format(
    LEARNING_RATE, BATCH_SIZE, WEIGHT_DECAY))

distilbert_train_result = distilbert_trainer.train()

16. ENSEMBLE (SOFT VOTING)

In [ ]:
import torch.nn.functional as F

def get_probs(trainer, dataset):
    raw_output = trainer.predict(dataset)
    logits = raw_output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    return probs

print("\nEvaluating on unseen test set...")

bert_test_probs = get_probs(bert_trainer, bert_test_dataset)
distilbert_test_probs = get_probs(distilbert_trainer, distilbert_test_dataset)


ensemble_probs = (bert_test_probs + distilbert_test_probs) / 2.0
ensemble_preds = np.argmax(ensemble_probs, axis=-1)

ensemble_accuracy = accuracy_score(test_labels, ensemble_preds)
ensemble_precision, ensemble_recall, ensemble_f1, _ = precision_recall_fscore_support(
    test_labels, ensemble_preds, average="weighted", zero_division=0
)

bert_solo_preds = np.argmax(bert_test_probs, axis=-1)
distilbert_solo_preds = np.argmax(distilbert_test_probs, axis=-1)

print("\nIndividual model test performance (for comparison):")
print("  BERT solo accuracy:      ", accuracy_score(test_labels, bert_solo_preds))
print("  DistilBERT solo accuracy:", accuracy_score(test_labels, distilbert_solo_preds))

17. FINAL RESULTS

In [ ]:
print("\n" + "=" * 70)
print("DATASET 1 - BBC TEXT - ENSEMBLE (BERT + DistilBERT) RESULTS")
print("=" * 70)

print(
    f"Learning Rate: {LEARNING_RATE} | "
    f"Batch Size: {BATCH_SIZE} | "
    f"Weight Decay: {WEIGHT_DECAY}"
)

print(
    f"Accuracy  : "
    f"{ensemble_accuracy * 100:.2f}%"
)

print(
    f"Precision : "
    f"{ensemble_precision * 100:.2f}%"
)

print(
    f"Recall    : "
    f"{ensemble_recall * 100:.2f}%"
)

print(
    f"F1 Score  : "
    f"{ensemble_f1 * 100:.2f}%"
)

print("=" * 70)